# Loan Approval Prediction
## Part 1: Baseline Models + EDA

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

try:
    from xgboost import XGBClassifier
except:
    !pip -q install xgboost
    from xgboost import XGBClassifier


## Load Dataset
Upload your CSV to Colab and update the filename if needed.

In [ ]:
df=pd.read_csv('/content/Loan(1).csv')
df.head()

## Dataset Information

In [ ]:
print(df.shape)
display(df.info())
display(df.describe())
display(df.isnull().sum().sort_values(ascending=False))

## Baseline Preprocessing

In [ ]:
df_base=df.copy()
drop_cols=[c for c in ['Unnamed: 0','id','year'] if c in df_base.columns]
df_base.drop(columns=drop_cols,inplace=True)

X=df_base.drop('status',axis=1)
y=df_base['status']

for c in X.select_dtypes(include=['int64','float64']).columns:
    X[c]=X[c].fillna(X[c].median())

for c in X.select_dtypes(include=['object']).columns:
    X[c]=X[c].fillna(X[c].mode()[0])

X=pd.get_dummies(X,drop_first=True)
X.columns=X.columns.astype(str).str.replace(r'[^a-zA-Z0-9_]','_',regex=True)

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)


## Train 3 Baseline Models

In [ ]:
models={
'Logistic Regression':LogisticRegression(max_iter=1000),
'Random Forest':RandomForestClassifier(random_state=42),
'XGBoost':XGBClassifier(random_state=42,eval_metric='logloss')
}

def evaluate(model):
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]
    return [accuracy_score(y_test,pred),
            precision_score(y_test,pred),
            recall_score(y_test,pred),
            f1_score(y_test,pred),
            roc_auc_score(y_test,prob)]

results={}
for n,m in models.items():
    results[n]=evaluate(m)

before_df=pd.DataFrame(results,index=['Accuracy','Precision','Recall','F1 Score','ROC AUC']).T
before_df


## Baseline Comparison

In [ ]:
before_df.plot(kind='bar',figsize=(8,5))
plt.title('Baseline Model Comparison')
plt.grid(axis='y',alpha=.3)
plt.xticks(rotation=0)
plt.show()

## Clean Dataset for EDA

In [ ]:
df_clean=df.copy()
df_clean.drop(columns=drop_cols,inplace=True)
for c in df_clean.select_dtypes(include=['int64','float64']).columns:
    df_clean[c]=df_clean[c].fillna(df_clean[c].median())
for c in df_clean.select_dtypes(include=['object']).columns:
    df_clean[c]=df_clean[c].fillna(df_clean[c].mode()[0])

## EDA

In [ ]:
sns.countplot(data=df_clean,x='status')
plt.title('Target Distribution')
plt.show()

nums=df_clean.select_dtypes(include=['int64','float64']).columns
df_clean[nums].hist(figsize=(18,15),bins=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,10))
sns.heatmap(df_clean[nums].corr(),cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

for col in ['income','loan_amount','property_value','rate_of_interest']:
    if col in df_clean.columns:
        plt.figure(figsize=(6,3))
        sns.boxplot(x=df_clean[col])
        plt.title(col)
        plt.show()
